# Synthetic artifact injection demo

This notebook walks through every artifact family `volumetric-qc` detects. We start from a clean synthetic volume, inject one artifact at a time, and watch the corresponding metric move.

This is the fastest way to develop intuition for what each check does and to calibrate thresholds for your own pipeline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from volumetric_qc import open_volume, run_qc, QCConfig
from volumetric_qc.synthetic import (
    clean_volume,
    inject_intensity_drift, inject_stripes, inject_bubbles,
    inject_bleed_through, inject_registration_shift,
    inject_clearing_residue, inject_focus_blur,
)
from volumetric_qc.metrics import (
    intensity, sharpness, artifacts, clearing, channel_bleed, registration
)

## 1. Build a clean baseline

Two channels, 32 z-slices, 192×192 in plane. ~400 neuron-like blobs. Background noise. No artifacts.

In [ ]:
vol_clean = clean_volume(shape=(2, 32, 192, 192), n_cells=400, background=100, signal=800, seed=0)
lv_clean = open_volume(vol_clean)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, z in enumerate([0, 16, 31]):
    axes[i].imshow(vol_clean[0, z], cmap='gray')
    axes[i].set_title(f'z={z}'); axes[i].axis('off')
fig.suptitle('Clean baseline volume — channel 0'); plt.tight_layout(); plt.show()

## 2. Intensity drift

Multiply each z-slice by a linear ramp from 0.75 → 1.25. The drift_slope metric should jump dramatically.

In [ ]:
vol_drift = inject_intensity_drift(vol_clean, slope=0.5)
res_clean = intensity.intensity_profile(open_volume(vol_clean).channel(0))
res_drift = intensity.intensity_profile(open_volume(vol_drift).channel(0))

print(f'clean drift_slope : {res_clean["drift_slope"]:+.3f}')
print(f'drifted drift_slope: {res_drift["drift_slope"]:+.3f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(res_clean['z'], res_clean['mean'], label='clean')
ax.plot(res_drift['z'], res_drift['mean'], label='drift')
ax.set_xlabel('z'); ax.set_ylabel('mean intensity'); ax.legend(); ax.grid(alpha=0.3); plt.show()

## 3. Focus blur on specific z-slices

Blur z=5..7 and z=28..30 with sigma=4. The sharpness metric should flag those exact slices.

In [ ]:
vol_blur = inject_focus_blur(vol_clean, z_indices=[5, 6, 7, 28, 29, 30], sigma=4.0)
res_blur = sharpness.sharpness_profile(open_volume(vol_blur).channel(0))
print('outlier slices:', res_blur['outlier_z'])

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(res_blur['z'], res_blur['relative'], 'o-')
for z in res_blur['outlier_z']:
    ax.axvline(z, color='red', alpha=0.3)
ax.set_xlabel('z'); ax.set_ylabel('relative Laplacian variance'); ax.grid(alpha=0.3); plt.show()

## 4. Stripe artifacts

Inject horizontal stripes in channel 0 only. The stripe energy ratio should jump for channel 0 but stay low for channel 1.

In [ ]:
vol_stripes = inject_stripes(vol_clean, amplitude=0.4, period=8, channel=0)
for name, vol in [('clean', vol_clean), ('striped', vol_stripes)]:
    res = artifacts.stripe_energy(open_volume(vol).channel(0), n_samples=5, tile_size=128)
    print(f'{name:8s} channel 0 mean ratio: {res["mean_ratio"]:.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(vol_stripes[0, 16], cmap='gray')
ax.set_title('z=16 of striped volume'); ax.axis('off'); plt.show()

## 5. Bubbles (air pockets / debris)

Punch dark circular regions into random z-slices. The blob detector should pick them up.

In [ ]:
vol_bubbles = inject_bubbles(vol_clean, n_bubbles=20, radius_range=(10, 18), channel=0, seed=2)
res = artifacts.bubble_count(open_volume(vol_bubbles).channel(0), n_samples=6)
print('counts per sampled slice:', res['counts'])
print('max:', res['max_per_slice'])

## 6. Cross-channel bleed-through

Add 40% of channel 0 into channel 1. The Pearson correlation on signal pixels jumps.

In [ ]:
vol_bleed = inject_bleed_through(vol_clean, source=0, target=1, factor=0.5)
for name, vol in [('clean', vol_clean), ('bled', vol_bleed)]:
    res = channel_bleed.bleed_through(open_volume(vol), n_samples=5)
    print(f'{name:8s} ch0->ch1 r = {res["pairwise_corr"]["ch0->ch1"]:.3f}')

## 7. Registration shift

Shift channel 1 by (3, 4) voxels. The detector should recover that shift.

In [ ]:
vol_shift = inject_registration_shift(vol_clean, channel=1, shift=(3, 4))
res = registration.cross_channel_shifts(open_volume(vol_shift), n_samples=5)
print('recovered shift ch0->ch1:', res['pairwise_shifts']['ch0->ch1'])

## 8. Clearing residue (RI mismatch speckle)

Add high-frequency speckle noise. The clearing_residue metric should rise.

In [ ]:
vol_residue = inject_clearing_residue(vol_clean, amplitude=0.6, scale=1.5, channel=0)
for name, vol in [('clean', vol_clean), ('residue', vol_residue)]:
    res = clearing.clearing_residue(open_volume(vol).channel(0), n_samples=5)
    print(f'{name:8s} speckle_energy: {res["speckle_energy"]:.3f}')

## 9. Full pipeline run on a multi-artifact volume

Combine every artifact above and let the full pipeline grade the volume.

In [ ]:
vol_bad = inject_intensity_drift(vol_clean, slope=0.4)
vol_bad = inject_focus_blur(vol_bad, z_indices=[5, 6, 28], sigma=3.5)
vol_bad = inject_stripes(vol_bad, amplitude=0.25, period=10, channel=0)
vol_bad = inject_bubbles(vol_bad, n_bubbles=12, channel=0, seed=0)
vol_bad = inject_bleed_through(vol_bad, source=0, target=1, factor=0.4)
vol_bad = inject_registration_shift(vol_bad, channel=1, shift=(3, 4))
vol_bad = inject_clearing_residue(vol_bad, amplitude=0.3, channel=0)

cfg = QCConfig()
cfg.sampling.fft_tile_size = 128
cfg.sampling.blob_z_sample = 6
result = run_qc(open_volume(vol_bad), cfg)
print(f'overall_pass: {result.overall_pass}')
print(f'n_fail: {result.n_fail}  n_warn: {result.n_warn}')
for f in result.flags:
    if f.severity != 'pass':
        print(f'  {f.severity:5s} {f.name:35s} value={f.value:.4f}  thr={f.threshold:.2f}')

## 10. Save the HTML dashboard

Self-contained HTML — open in any browser, no server needed.

In [ ]:
from volumetric_qc.reports import write_html_report, write_json_summary
write_html_report(result, 'synthetic_demo.html', title='Synthetic artifact demo')
write_json_summary(result, 'synthetic_demo.json')